# Model Collapse Experiment

Same setup as the original multi-generation experiment: two pipelines (synthetic-only training vs human+synthetic training), 10 generations each. We run the full experiment **4 times** with different random seeds, then report **mean and std** for all evaluations.

- **Pipeline A**: Human prompts from **articles.json** (randomly select and truncate to 40 tokens once per run; same prompts for all generations in that run). Train on **synthetic-only** (120 tokens).
- **Pipeline B**: Human prompts from **human_40.json** (exact 40-token texts; used for all runs and generations). Train on **human + synthetic** (160 tokens).
- **Evaluation** (same texts for all 4 runs): **Perplexity** = `dewiki_medical_250tokens_1001_2000.json`; **NTP** = `dewiki_medical_50tokens.json`. MCQ from CSV if provided.
- **Save**: All metrics, generated synthetic/mixed data, model checkpoints, and summary mean/std.

## 1. Small-batch switch (run small first to verify, then set False for full run)

In [ ]:
# Set to True to run a small batch (few runs, few generations, fewer docs).
# Set to False for the full experiment (4 runs, 10 generations, full doc counts).
SMALL_RUN = False

if SMALL_RUN:
    N_RUNS = 2
    N_GENERATIONS = 2
    NUM_HUMAN_DOCS = 100
    NUM_SYNTHETIC_PER_GEN = 100
    EVAL_PERPLEXITY_DOCS = 50
    EVAL_PERPLEXITY_TOKENS = 250
    EVAL_NTP_DOCS = 20
    EVAL_NTP_TOKENS = 40
    MCQ_SAMPLE = 100
else:
    N_RUNS = 4
    N_GENERATIONS = 10
    NUM_HUMAN_DOCS = 800
    NUM_SYNTHETIC_PER_GEN = 800
    EVAL_PERPLEXITY_DOCS = 1000
    EVAL_PERPLEXITY_TOKENS = 250
    EVAL_NTP_DOCS = 1000
    EVAL_NTP_TOKENS = 40
    MCQ_SAMPLE = None  # use full CSV

RUN_SEEDS = [42, 43, 44, 45][:N_RUNS]
print("SMALL_RUN =", SMALL_RUN)
print("N_RUNS =", N_RUNS, "| N_GENERATIONS =", N_GENERATIONS)
print("EVAL: perplexity", EVAL_PERPLEXITY_DOCS, "docs,", EVAL_PERPLEXITY_TOKENS, "tokens | NTP", EVAL_NTP_DOCS, "docs,", EVAL_NTP_TOKENS, "tokens")

## 2. Imports and paths

In [ ]:
%pip install unsloth transformers accelerate bitsandbytes

import os
import sys
import json
import gc
import random

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

from unsloth import FastLanguageModel
from datasets import Dataset, concatenate_datasets
from transformers import TrainingArguments, DataCollatorForLanguageModeling, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import matplotlib.pyplot as plt

# Project root and all combied for imports
ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if "all combied" in os.getcwd() else os.getcwd()
COMBINED = os.path.join(ROOT, "all combied")
if COMBINED not in sys.path:
    sys.path.insert(0, COMBINED)

from exp_config import (
    BASE_MODEL,
    REAL_TOKENS,
    SYNTHETIC_TOKENS,
    TOTAL_TOKENS,
    NUM_TRAIN_EPOCHS,
    LEARNING_RATE,
    set_seed,
)
# Use model_collapse experiment folder (no other naming)
OUTPUT_DIR = "experiment_model_collapse"
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
DATA_DIR = os.path.join(OUTPUT_DIR, "data")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
from exp_data import (
    load_articles_json,
    prepare_human_dataset,
    load_human40_json,
    save_human_dataset,
)
from exp_generation import generate_synthetic_docs, generate_mixed_docs
from exp_training import finetune_on_accumulated_data, finetune_on_mixed_data
import pipeline_eval
import mcq_eval

from unsloth import FastLanguageModel

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = ""  # set your token

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Required files and data load (eval texts fixed for all 4 runs)

In [ ]:
# Required file paths (under ROOT or cwd)
ARTICLES_JSON = os.path.join(ROOT, "articles.json")
HUMAN_40_JSON = os.path.join(ROOT, "human_40.json")
DEWIKI_250_JSON = os.path.join(ROOT, "dewiki_medical_250tokens_1001_2000.json")
DEWIKI_50_JSON = os.path.join(ROOT, "dewiki_medical_50tokens.json")

REQUIRED_FILES = {
    "articles.json (Pipeline A)": ARTICLES_JSON,
    "human_40.json (Pipeline B)": HUMAN_40_JSON,
    "dewiki_250 (perplexity eval)": DEWIKI_250_JSON,
    "dewiki_50 (NTP eval)": DEWIKI_50_JSON,
}
missing = [name for name, path in REQUIRED_FILES.items() if not os.path.exists(path)]
if missing:
    print("Missing required files:", missing)
    print("Add these files and re-run. Experiment will not start.")
    FILES_OK = False
else:
    print("All required files found.")
    FILES_OK = True

In [ ]:
# Load data only if files OK. Eval texts are loaded once and never change (same for all 4 runs).
articles = None
human_docs_pipeline_b = None
perplexity_eval_texts = None
NTP_EVAL_JSON = None

if FILES_OK:
    articles = load_articles_json(ARTICLES_JSON)
    print("Pipeline A: Loaded", len(articles), "articles (will sample per run with run seed).")

    human_docs_pipeline_b = load_human40_json(HUMAN_40_JSON)
    print("Pipeline B: Loaded", len(human_docs_pipeline_b), "human 40-token prompts (same for all runs and gens).")

    with open(DEWIKI_250_JSON, "r", encoding="utf-8") as f:
        dewiki_250 = json.load(f)
    perplexity_eval_texts = [d["text"] for d in dewiki_250 if isinstance(d, dict) and d.get("text")]
    if not perplexity_eval_texts:
        perplexity_eval_texts = [s for s in dewiki_250 if isinstance(s, str) and s.strip()]
    perplexity_eval_texts = perplexity_eval_texts[:EVAL_PERPLEXITY_DOCS]
    print("Perplexity eval:", len(perplexity_eval_texts), "texts (fixed for all runs).")

    NTP_EVAL_JSON = DEWIKI_50_JSON
    print("NTP eval: using", NTP_EVAL_JSON, "(max_prompts=", EVAL_NTP_DOCS, ").")

    eval_config = {
        "perplexity_file": DEWIKI_250_JSON,
        "perplexity_ndocs": len(perplexity_eval_texts),
        "ntp_file": NTP_EVAL_JSON,
        "ntp_max_prompts": EVAL_NTP_DOCS,
    }
    with open(os.path.join(DATA_DIR, "eval_config.json"), "w", encoding="utf-8") as f:
        json.dump(eval_config, f, indent=2)
    print("Saved eval_config.json to", DATA_DIR)

if FILES_OK:
    print("Loading base model for tokenizer (Pipeline A needs it to build human prompts from articles)...")
    model_for_setup, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
        token=os.environ.get("HF_TOKEN"),
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("Tokenizer ready.")
    del model_for_setup
    clear_gpu()
else:
    tokenizer = None

## 4. Load MCQ dataset (optional)

In [ ]:
MCQ_CSV = os.path.join(ROOT, "hs1_hs2.csv")
if not os.path.exists(MCQ_CSV):
    MCQ_CSV = "hs1_hs2.csv"

df_mcq = None
if os.path.exists(MCQ_CSV):
    df_mcq = pd.read_csv(MCQ_CSV)
    if MCQ_SAMPLE is not None:
        df_mcq = df_mcq.sample(n=min(MCQ_SAMPLE, len(df_mcq)), random_state=42).copy()
    print("MCQ loaded:", len(df_mcq), "questions")
else:
    print("MCQ CSV not found; MCQ evaluation will be skipped.")

## 5. Evaluation helpers (perplexity mean, NTP, MCQ)

In [ ]:
def eval_perplexity_mean(model, tokenizer, texts, max_docs=None):
    """Average perplexity over texts (each truncated to tokenizer max in pipeline_eval.model_perplexity)."""
    use = texts[:max_docs] if max_docs else texts
    ppls = []
    for t in tqdm(use, desc="Perplexity", leave=False):
        ppl = pipeline_eval.model_perplexity(model, tokenizer, t)
        ppls.append(ppl)
    return float(np.mean(ppls)) if ppls else 0.0

def eval_ntp_gini(model, tokenizer, json_path, max_prompts=None):
    out = pipeline_eval.evaluate_ntp_gini_collapsed(model, tokenizer, json_path, max_prompts=max_prompts)
    return out

def run_mcq_eval(model, tokenizer, df, model_name):
    if df is None or len(df) == 0:
        return None
    results_df, accuracy, _ = mcq_eval.evaluate_model(model, tokenizer, df, model_name, use_chat_template=False)
    return accuracy

## 6. Main loop: 4 runs x 2 pipelines x 10 generations

In [ ]:
# Storage: list of dicts with run_id, pipeline, gen, metrics. Save everything (metrics, generated data, models).
all_results = []

if not FILES_OK:
    print("Skipping main loop: required files missing.")
else:
    for run_id, seed in enumerate(RUN_SEEDS):
        set_seed(seed)
        # Pipeline A: human prompts from articles (same set for all generations in this run)
        human_docs_a = prepare_human_dataset(
            articles, tokenizer, num_docs=NUM_HUMAN_DOCS, num_tokens=REAL_TOKENS, seed=seed
        )
        run_dir = os.path.join(MODELS_DIR, f"run_{run_id}")
        os.makedirs(run_dir, exist_ok=True)
        save_human_dataset(human_docs_a, os.path.join(run_dir, "human_dataset_pipelineA.json"))

        for pipeline_name, use_mixed in [("A", False), ("B", True)]:
            # Pipeline A: human_docs_a (same set all generations in this run).
            # Pipeline B: human_40.json, but we want DIFFERENT human texts per generation.
            # We shuffle once per run, then use a sliding window across generations.
            if use_mixed:
                human_pool = human_docs_pipeline_b.copy()
                random.Random(seed).shuffle(human_pool)
            else:
                human_pool = human_docs_a
            pipe_dir = os.path.join(run_dir, f"pipeline_{pipeline_name}")
            os.makedirs(pipe_dir, exist_ok=True)

            for gen in range(N_GENERATIONS):
                clear_gpu()
                if gen == 0:
                    current_model, current_tokenizer = FastLanguageModel.from_pretrained(
                        model_name=BASE_MODEL,
                        max_seq_length=2048,
                        dtype=None,
                        load_in_4bit=True,
                        token=os.environ.get("HF_TOKEN"),
                    )
                else:
                    prev_dir = os.path.join(pipe_dir, f"M{gen}")
                    try:
                        current_model, current_tokenizer = FastLanguageModel.from_pretrained(
                            model_name=prev_dir,
                            max_seq_length=2048,
                            dtype=None,
                            load_in_4bit=True,
                            token=os.environ.get("HF_TOKEN"),
                        )
                    except Exception:
                        from peft import PeftModel
                        current_model, current_tokenizer = FastLanguageModel.from_pretrained(
                            model_name=BASE_MODEL, max_seq_length=2048, dtype=None, load_in_4bit=True,
                            token=os.environ.get("HF_TOKEN"),
                        )
                        current_model = PeftModel.from_pretrained(current_model, prev_dir)
                        current_model = current_model.merge_and_unload()

                if current_tokenizer.pad_token is None:
                    current_tokenizer.pad_token = current_tokenizer.eos_token
                FastLanguageModel.for_inference(current_model)

                model_label = f"run{run_id}_{pipeline_name}_M{gen}"
                metrics = {"run_id": run_id, "pipeline": pipeline_name, "gen": gen}

                # Eval (same texts for all 4 runs)
                ppl = eval_perplexity_mean(current_model, current_tokenizer, perplexity_eval_texts, max_docs=EVAL_PERPLEXITY_DOCS)
                metrics["perplexity_mean"] = ppl
                ntp_out = eval_ntp_gini(current_model, current_tokenizer, NTP_EVAL_JSON, max_prompts=EVAL_NTP_DOCS)
                metrics["mean_gini_top100"] = ntp_out.get("mean_gini_top100", 0)
                metrics["collapsed_rate"] = ntp_out.get("collapsed_rate", 0)
                if df_mcq is not None:
                    acc = run_mcq_eval(current_model, current_tokenizer, df_mcq, model_label)
                    metrics["mcq_accuracy"] = acc

                all_results.append(metrics)

                # Choose human docs for this generation.
                if use_mixed:
                    total = len(human_pool)
                    start = (gen * NUM_SYNTHETIC_PER_GEN) % max(total, 1)
                    end = start + NUM_SYNTHETIC_PER_GEN
                    if end <= total:
                        human_docs_this = human_pool[start:end]
                    else:
                        part1 = human_pool[start:total]
                        part2 = human_pool[0:(end - total)]
                        human_docs_this = part1 + part2
                else:
                    human_docs_this = human_pool

                # Generate: A = synthetic only (from human_docs_a), B = mixed (from human_40, different slice per gen)
                if use_mixed:
                    train_docs = generate_mixed_docs(
                        current_model, current_tokenizer, human_docs_this, gen,
                        NUM_SYNTHETIC_PER_GEN, REAL_TOKENS, SYNTHETIC_TOKENS, seed=seed
                    )
                else:
                    train_docs = generate_synthetic_docs(
                        current_model, current_tokenizer, human_docs_this, gen,
                        NUM_SYNTHETIC_PER_GEN, REAL_TOKENS, SYNTHETIC_TOKENS, seed=seed
                    )

                syn_path = os.path.join(pipe_dir, f"D{gen}_synthetic.json") if not use_mixed else os.path.join(pipe_dir, f"D{gen}_mixed.json")
                with open(syn_path, "w", encoding="utf-8") as f:
                    json.dump(train_docs, f, ensure_ascii=False, indent=2)

                next_dir = os.path.join(pipe_dir, f"M{gen + 1}")
                if use_mixed:
                    current_model = finetune_on_mixed_data(current_model, current_tokenizer, train_docs, next_dir, gen, lr=LEARNING_RATE, num_epochs=NUM_TRAIN_EPOCHS)
                else:
                    current_model = finetune_on_accumulated_data(current_model, current_tokenizer, train_docs, next_dir, gen, lr=LEARNING_RATE, num_epochs=NUM_TRAIN_EPOCHS)

                del current_model, current_tokenizer
                clear_gpu()

    print("Done all runs. Saved: metrics (in all_results), generated D* files, model checkpoints M*.")

## 7. Save raw results and compute mean / std per (pipeline, gen)

In [ ]:
if not all_results:
    print("No results to save (missing files or loop not run).")
else:
    results_df = pd.DataFrame(all_results)
    raw_path = os.path.join(DATA_DIR, "model_collapse_raw_results.csv")
    results_df.to_csv(raw_path, index=False)
    with open(os.path.join(DATA_DIR, "model_collapse_raw_results.json"), "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2)
    print("Saved raw results to", raw_path, "and .json")

    agg = results_df.groupby(["pipeline", "gen"]).agg(
        perplexity_mean_mean=("perplexity_mean", "mean"),
        perplexity_mean_std=("perplexity_mean", "std"),
        mean_gini_mean=("mean_gini_top100", "mean"),
        mean_gini_std=("mean_gini_top100", "std"),
        collapsed_rate_mean=("collapsed_rate", "mean"),
        collapsed_rate_std=("collapsed_rate", "std"),
    ).reset_index()
    if "mcq_accuracy" in results_df.columns:
        mcq_agg = results_df.groupby(["pipeline", "gen"])["mcq_accuracy"].agg(["mean", "std"]).reset_index()
        mcq_agg.columns = ["pipeline", "gen", "mcq_accuracy_mean", "mcq_accuracy_std"]
        agg = agg.merge(mcq_agg, on=["pipeline", "gen"])

    summary_path = os.path.join(DATA_DIR, "model_collapse_summary_mean_std.csv")
    agg.to_csv(summary_path, index=False)
    print("Saved summary (mean/std) to", summary_path)
    display(agg)